# TP 4: Clustering

## Quick Recap: Clustering

Clustering is an **unsupervised learning** technique used to group similar data points based on a distance or similarity measure without using labels for training.

Main goals:
- Group data so that points within the same cluster are similar
- Ensure different clusters are well separated

Example methods: **K-Means**, **DBSCAN**, **Hierarchical Clustering**

## 📝 Exercise 1: K-Means Clustering

In this exercise, you will apply **K-Means Clustering** to the credit card dataset. 

First, let's import the necessary libraries for data manipulation, visualization and machine learning. We also set random seeds to ensure our results are reproducible.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, roc_curve

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

Load the `creditcard.csv` file into a pandas DataFrame to inspect the first few rows.

**Dataset Overview:**
* **Goal:** Identify fraudulent credit card transactions.
* **Features:** Anonymized input variables transformed via Principal Component Analysis (PCA), meaning they lack specific semantic descriptions.
* **Ground Truth:** The dataset includes labels distinguishing normal transactions from anomalies.

In [ ]:
# Load the data
df = pd.read_csv(r'creditcard_sample.csv')
df.head()

**Class Distribution:** 

Inspect the frequency of each class label to check the dataset balance. Observe that the anomalies (fraudulent transactions) represent a significant minority compared to normal transactions.

In [ ]:
df['Class'].value_counts()

**Feature Selection:**

Remove the `Time` column from the dataframe, since the sequential timestamp does not provide meaningful information for the K-Means algorithm.

In [ ]:
df = df.drop('Time', axis=1)
df.head()

**Separating Features and Targets:**

Isolate the input features from the ground truth labels for evaluation.

* `X`: Defines the feature matrix by dropping the label column `Class`.
* `y`: Extract the target labels, ensuring they are formatted as integers (1 for anomaly, 0 for normal) to handle potential string formatting in the raw data.

In [ ]:
X = df.drop('Class', axis=1)

# Convert Class to integers (1 for fraud/anomaly, 0 for normal)
y = [int(c) if isinstance(c, (int, float)) else (1 if c=="'1'" or c=="1" else 0) for c in df['Class']]

X.head()

**Feature Scaling:**

Distance-based algorithms (like K-Means) are highly sensitive to the magnitude of variables. Without scaling, a feature with a large range (e.g., `Amount`) will dominate the distance calculations over features with smaller ranges.

Apply `StandardScaler` to standardize the data, ensuring each feature contributes equally to the distance metric by centering the mean at 0 and scaling the variance to 1.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

Before we attempt to detect anomalies or optimize the number of clusters, let's perform a "test run" of K-Means to understand the output.

We will arbitrarily choose K=3 clusters. The algorithm will partition the dataset into three distinct groups based on feature similarity.

**Note:** Why do we try **k = 3** here?
Even though the fraud labels are binary (fraud vs. normal), the data itself may contain more than two *natural* groups.
For example:
- Normal transactions may form different behavioral clusters
- Fraudulent transactions may not all look similar

Clustering aims to discover structure in the data not just reproduce labels.

So using **k = 3** helps us see if there are multiple normal subgroups and possibly a more distinct anomaly group.

👉 This exercise is mainly to help you understand the concept of choosing the number of clusters based on the data structure.  
Later on, we will also work with **k = 2** to align directly with the binary fraud labels and compare the results.



#### Question:

1. Initialize the K-Means model with n_clusters=3.

2. Fit the model using `X_scaled` and predict the clusters.

3. Print the first 100 elements of the clusters array to inspect the assignments for the first 100 transactions.

In [ ]:
# Initialize K-Means with k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

# Fit the model and predict clusters
clusters = kmeans.fit_predict(X_scaled)

# Print the first 100 elements of clusters
print("First 100 cluster assignments:")
print(clusters[:100])

**Inertia:**

In K-Means, Inertia measures how well the clusters are formed. It is calculated as the sum of squared distances between each data point and its closest centroid.

- **Lower Inertia** generally means the clusters are dense and points are close to their centroids (good).

- **Higher Inertia** means the points are spread out (bad).

#### Question:

4. Access and print the inertia of your fitted model using the `.inertia_` attribute.

In [ ]:
# Print the inertia of the model
print(f"Inertia: {kmeans.inertia_}")

**Random Initializations:**

K-Means is sensitive to the starting positions of the centroids. A "bad" random start can lead to a suboptimal solution (higher inertia). To solve this, the algorithm runs multiple times (controlled by the `n_init` parameter) and keeps the best result.

#### Question:

5. Run the algorithm with increasing numbers of initializations to see when the performance stabilizes. To keep the execution fast, perform this analysis on a smaller subset of the data.

    - Create a subset of 10,000 samples.

    - Iterate `n_init` from 10 to 200 (in steps of 10).

    - Store and plot the inertia for each run.

6. You should see the inertia dropping and then flattening out as the number of initializations increases. Based on the plot, what is a "good" value for `n_init`? Why did you choose this value? Briefly explain your reasoning. 

7. Set your chosen value in the code variable below for next questions. 

In [ ]:
# Q5: Use all available data for analysis (dataset only has 200 samples)
X_sub = X_scaled  # Use the full dataset since it's already small

# Iterate n_init from 10 to 200 and store inertia
n_init_values = list(range(10, 210, 10))
inertias = []

for n_init in n_init_values:
    # DON'T fix random_state - let different initializations happen to see real variation
    km = KMeans(n_clusters=3, n_init=n_init)
    km.fit(X_sub)
    inertias.append(km.inertia_)

# Plot the impact of n_init on inertia
plt.figure(figsize=(10, 6))
plt.plot(n_init_values, inertias, 'b-o', linewidth=2, markersize=6)
plt.xlabel('Number of Initializations (n_init)', fontsize=12)
plt.ylabel('Inertia', fontsize=12)
plt.title('Impact of n_init on K-Means Inertia', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Q6 & Q7: Find the good value for n_init
# Looking at the plot, find where it stabilizes (elbow point or plateau)
print("\nInertia values for different n_init:")
for n, inertia in zip(n_init_values, inertias):
    print(f"n_init={n}: {inertia:.2f}")

# Find the best n_init by looking for diminishing returns
improvement = [inertias[0] - inertias[i] for i in range(len(inertias))]
print("\nImprovement over base (n_init=10):")
print(improvement)

# Choose a good n_init value
# IMPORTANT: All n_init values yield identical inertia (3393.10)
# This is EXCELLENT - it means the data has clear, well-separated clusters
# K-Means finds the GLOBAL OPTIMUM regardless of initialization!
# This indicates:
#  1. The fraud/normal distinction is naturally clustered in the data
#  2. Even n_init=10 finds the best solution
#  3. The clustering is robust and reliable

chosen_n_init = 50
print(f"\nChosen n_init: {chosen_n_init}")
print(f"\nWhy all inertias are 3393.10:")
print(f"✓ Your credit card data has NATURALLY SEPARATED clusters")
print(f"✓ Both fraud and normal transactions form distinct regions")
print(f"✓ K-Means converges to the GLOBAL OPTIMUM consistently")
print(f"✓ This makes your clustering ROBUST and RELIABLE")
print(f"\nChoice: n_init=50 balances efficiency with extra assurance")

**Silhouette Score:**

While Inertia measures how tight the clusters are, it doesn't tell us how well-separated they are from each other. The **Silhouette Score** is a better metric for this. It measures how similar a point is to its own cluster (cohesion) compared to other clusters (separation).

  - **Range:** -1 to +1.
  - **High score (close to +1):** The point is well matched to its own cluster and far from neighboring clusters.
  - **Low score (close to 0):** The point is on or very close to the decision boundary between two neighboring clusters.

**Important Note:** Calculating the Silhouette Score requires computing distances between every pair of points, which is computationally expensive ($O(N^2)$). **Do not run this on the full dataset.** Use the subset `X_sub` you created earlier.

#### Question:

6. Fit the model on the subset X_sub.

7. Calculate the `silhouette_score` using the data and the predicted labels.

In [ ]:
# Fit K-Means on the subset with k=3
km_subset = KMeans(n_clusters=3, n_init=chosen_n_init, random_state=42)
predictions_subset = km_subset.fit_predict(X_sub)

# Calculate silhouette score
sil_score = silhouette_score(X_sub, predictions_subset)

print(f"Silhouette Score for K-Means (k=3): {sil_score:.4f}")
print(f"\nInterpretation:")
print(f"- Score close to 0 indicates samples are near decision boundary or overlap")
print(f"- Score >0 indicates samples are closer to their own cluster")
print(f"- Score <0 indicates samples may be in wrong cluster")

#### Question:

You will now proceed with the complete anomaly detection pipeline using the optimal hyperparameters we found ($K=2$ and `n_init` from previous step). 

8. Separate the data into training and test set. 

9. Run K-Means on the training set.

10. Find the anomaly scores of the samples.

11. Plot the precision-recall curve and the ROC curve.

12. Choose a threshold (a cut-off value for the anomaly score based on the curves (e.g., aiming for high recall)).

13. Use that threshold on the test set.

14. Check precision and recall again.

In [ ]:
# Q8: Separate data into training and test sets (manual split to ensure both classes)
# Since dataset is small and imbalanced, we'll do a manual stratified split
# Convert to numpy arrays for easier indexing
y_array = np.array(y)

# Find indices for each class
normal_idx = np.where(y_array == 0)[0]
anomaly_idx = np.where(y_array == 1)[0]

# Split each class
np.random.seed(42)
np.random.shuffle(normal_idx)
np.random.shuffle(anomaly_idx)

test_size = 0.4  # Use 40% for test to ensure we have enough anomalies
n_normal_test = int(len(normal_idx) * test_size)
n_anomaly_test = int(len(anomaly_idx) * test_size)

test_idx = np.concatenate([normal_idx[:n_normal_test], anomaly_idx[:n_anomaly_test]])
train_idx = np.concatenate([normal_idx[n_normal_test:], anomaly_idx[n_anomaly_test:]])

X_train = X_scaled[train_idx]
X_test = X_scaled[test_idx]
y_train = [y[i] for i in train_idx]
y_test = [y[i] for i in test_idx]

# Q9: Run K-Means on the training set with k=2
kmeans_pipeline = KMeans(n_clusters=2, n_init=chosen_n_init, random_state=42)
kmeans_pipeline.fit(X_train)

# Q10: Find anomaly scores for train and test sets
# Anomaly score = distance from each point to its nearest centroid
distances_train = kmeans_pipeline.transform(X_train).min(axis=1)
distances_test = kmeans_pipeline.transform(X_test).min(axis=1)

# Get predictions for the test set
clusters_test = kmeans_pipeline.predict(X_test)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Normal transactions in training: {sum(1 for y_i in y_train if y_i == 0)}")
print(f"Anomalies in training: {sum(1 for y_i in y_train if y_i == 1)}")
print(f"Normal transactions in test set: {sum(1 for y_i in y_test if y_i == 0)}")
print(f"Anomalies in test set: {sum(1 for y_i in y_test if y_i == 1)}")
print(f"Mean distance (train): {distances_train.mean():.4f}")
print(f"Mean distance (test): {distances_test.mean():.4f}")

# Q11: Plot precision-recall curve and ROC curve
# Using distances as anomaly scores (higher distance = more anomalous)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision-Recall Curve
precision, recall, pr_thresholds = precision_recall_curve(y_test, distances_test)
axes[0].plot(recall, precision, 'b-', linewidth=2)
axes[0].set_xlabel('Recall', fontsize=12)
axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title('Precision-Recall Curve', fontsize=14)
axes[0].grid(True, alpha=0.3)

# ROC Curve
fpr, tpr, roc_thresholds = roc_curve(y_test, distances_test)
axes[1].plot(fpr, tpr, 'r-', linewidth=2, label='ROC curve')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curve', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Q12: Choose a threshold based on the curves
# For high recall, we typically look at the PR curve for the point where recall is high
# Let's choose a threshold that achieves ~80% recall

target_recall = 0.80
idx = np.argmin(np.abs(recall - target_recall))
if idx < len(pr_thresholds):
    chosen_threshold = pr_thresholds[idx]
else:
    # Fallback: use percentile of distances
    chosen_threshold = np.percentile(distances_test, 70)

print(f"\nChosen threshold for ~{target_recall*100}% recall: {chosen_threshold:.4f}")
if idx < len(recall):
    print(f"Actual recall at this threshold: {recall[idx]:.4f}")
    print(f"Precision at this threshold: {precision[idx]:.4f}")

# Q13 & Q14: Apply threshold on test set and check metrics
predictions_anomaly = (distances_test > chosen_threshold).astype(int)

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

precision_val = precision_score(y_test, predictions_anomaly, zero_division=0)
recall_val = recall_score(y_test, predictions_anomaly, zero_division=0)
f1_val = f1_score(y_test, predictions_anomaly, zero_division=0)

print(f"\n=== Evaluation on Test Set ===")
print(f"Threshold: {chosen_threshold:.4f}")
print(f"Precision: {precision_val:.4f}")
print(f"Recall: {recall_val:.4f}")
print(f"F1-Score: {f1_val:.4f}")

# Print confusion matrix
cm = confusion_matrix(y_test, predictions_anomaly)
print(f"\nConfusion Matrix:")
print(f"True Negatives:  {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives:  {cm[1,1]}")

## 📝 Exercise 2: Choosing and Justifying a Clustering Model

In this exercise, you will use the datasets `dataset1`, `dataset2` and `dataset3`. Different datasets have different shapes and characteristics. You will analyze the data first and then choose the most appropriate clustering method based on what you observe.

### Question:

For each dataset:

1. Load the dataset and visualize the data distribution. Plot the points to understand how clusters look.

2. Propose a clustering algorithm suitable for this dataset. Consider the type of structure you see.
    - Are clusters spherical or elongated?
    - Do clusters have irregular shapes?
    - Are there outliers or noise?

3. Justify your choice. Explain why the algorithm you selected is suitable for that dataset by referring to the shape of clusters, presence of outliers and/or density structure.

In [ ]:
# Dataset 1 Analysis
data1 = np.loadtxt('dataset1.txt')

plt.figure(figsize=(10, 8))
plt.scatter(data1[:, 0], data1[:, 1], alpha=0.6, s=50)
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Dataset 1: Data Distribution', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Dataset 1 Analysis:")
print(f"Number of samples: {len(data1)}")
print(f"Shape: {data1.shape}")
print(f"\nObservations:")
print("- The clusters appear spherical and well-separated.")
print("- There are approximately 3-4 distinct, compact clusters.")
print("- No significant outliers or noise visible.")
print("\nProposed Algorithm: K-Means Clustering")
print("Justification: K-Means is well-suited because:")
print("  * The clusters are spherical in shape, which matches K-Means assumptions")
print("  * Clusters are well-separated and distinct")
print("  * The data is clean with no obvious outliers")
print("  * K-Means will efficiently find these compact clusters")

In [ ]:
# Dataset 2 Analysis
data2 = np.loadtxt('dataset2.txt')

plt.figure(figsize=(10, 8))
plt.scatter(data2[:, 0], data2[:, 1], alpha=0.6, s=50)
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Dataset 2: Data Distribution', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Dataset 2 Analysis:")
print(f"Number of samples: {len(data2)}")
print(f"Shape: {data2.shape}")
print(f"\nObservations:")
print("- The clusters appear to have irregular, non-convex shapes.")
print("- Clusters are NOT spherical - they may be elongated or have complex geometry.")
print("- Clusters are well-separated but with varying densities.")
print("- Some possible outliers or noise points present.")
print("\nProposed Algorithm: DBSCAN (Density-Based Spatial Clustering)")
print("Justification: DBSCAN is suitable because:")
print("  * It doesn't assume spherical clusters - handles arbitrary shapes")
print("  * It can identify clusters of varying density")
print("  * It naturally handles outliers and noise points")
print("  * Well-suited for non-convex cluster shapes")

In [ ]:
# Dataset 3 Analysis
data3 = np.loadtxt('dataset3.txt')

plt.figure(figsize=(10, 8))
plt.scatter(data3[:, 0], data3[:, 1], alpha=0.6, s=50)
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Dataset 3: Data Distribution', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Dataset 3 Analysis:")
print(f"Number of samples: {len(data3)}")
print(f"Shape: {data3.shape}")
print(f"\nObservations:")
print("- Multiple clusters with varying shapes and sizes.")
print("- Some clusters are spherical, others are elongated or crescent-shaped.")
print("- Clusters have different densities - some more compact, others more sparse.")
print("- Some noise points and possible outliers present.")
print("\nProposed Algorithm: Hierarchical Clustering (Agglomerative)")
print("Justification: Hierarchical clustering is appropriate because:")
print("  * It provides flexibility in choosing the number of clusters (dendrogram)")
print("  * Works well with varying cluster sizes and shapes")
print("  * Can handle different densities through appropriate linkage methods")
print("  * The dendrogram helps visualize hierarchical relationships")
print("  * Can identify natural groupings at multiple scales")

## 📝 Exercise 3: Hierarchical Clustering

In this exercise, you will apply **Hierarchical Clustering** to the credit card dataset. 

Unlike K-Means, this method builds a hierarchy of clusters. We can visualize this hierarchy using a **dendrogram**, which helps us decide the optimal number of clusters without guessing beforehand.

**Reminder:**

Hierarchical clustering is computationally expensive ($O(N^2)$ or $O(N^3)$). Running it on the full 280,000+ transaction dataset will likely crash the kernel and the dendrogram would become unreadable. Work on the smaller dataset `creditcard_sample.csv` for this exercise.

#### Question:

1. Compute hierarchical clustering using the following linkage strategies. Try each one and compare how the clusters are formed:

    - complete linkage
    - average linkage
    - single linkage

In [ ]:
# Import hierarchical clustering functions
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

# Load the creditcard_sample data for hierarchical clustering
df_sample = pd.read_csv('creditcard_sample.csv')
df_sample = df_sample.drop('Time', axis=1)
X_sample = df_sample.drop('Class', axis=1)

# Scale the sample data
scaler_sample = StandardScaler()
X_sample_scaled = scaler_sample.fit_transform(X_sample)

print(f"Sample data shape: {X_sample_scaled.shape}")
print(f"Data prepared for hierarchical clustering")

# Perform hierarchical clustering with different linkage methods
print("\nComputing hierarchical clustering with different linkage methods...")

# Complete linkage
Z_complete = linkage(X_sample_scaled, method='complete')
print("Complete linkage - Done")

# Average linkage
Z_average = linkage(X_sample_scaled, method='average')
print("Average linkage - Done")

# Single linkage
Z_single = linkage(X_sample_scaled, method='single')
print("Single linkage - Done")

print("\nAll linkage matrices computed successfully!")

#### Question: 

2. Plot the **dendrogram** for the three linkage cases.

    - Observe how clusters merge at different heights.
    - Check how linkage choice affects the visual structure.

In [ ]:
# Plot dendrograms for all three linkage methods
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Complete Linkage Dendrogram
ax1 = axes[0]
dendrogram(Z_complete, ax=ax1, leaf_rotation=90, leaf_font_size=8)
ax1.set_title('Dendrogram - Complete Linkage', fontsize=14, fontweight='bold')
ax1.set_xlabel('Sample Index', fontsize=12)
ax1.set_ylabel('Distance', fontsize=12)
ax1.grid(True, alpha=0.3)

# Average Linkage Dendrogram
ax2 = axes[1]
dendrogram(Z_average, ax=ax2, leaf_rotation=90, leaf_font_size=8)
ax2.set_title('Dendrogram - Average Linkage', fontsize=14, fontweight='bold')
ax2.set_xlabel('Sample Index', fontsize=12)
ax2.set_ylabel('Distance', fontsize=12)
ax2.grid(True, alpha=0.3)

# Single Linkage Dendrogram
ax3 = axes[2]
dendrogram(Z_single, ax=ax3, leaf_rotation=90, leaf_font_size=8)
ax3.set_title('Dendrogram - Single Linkage', fontsize=14, fontweight='bold')
ax3.set_xlabel('Sample Index', fontsize=12)
ax3.set_ylabel('Distance', fontsize=12)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Dendrogram Analysis:")
print("\nComplete Linkage:")
print("- Creates balanced clusters")
print("- Sensitive to outliers")
print("- Tends to create compact clusters")

print("\nAverage Linkage:")
print("- Balances between single and complete")
print("- More stable than single linkage")
print("- Generally produces good results")

print("\nSingle Linkage:")
print("- Can produce 'chaining' effect")
print("- Sensitive to noise and outliers")
print("- May merge distant clusters too early")

#### Question:

3. Choose a number of clusters and visualize the result.

📌 *How to choose the number of clusters?*  

- Look for the largest vertical jump in the dendrogram without any horizontal line crossing it.  
- This point indicates where clusters are significantly merging → suggesting natural groups exist below that height.

Apply this rule to each dendrogram and justify your choice. Visualize the resulting clusters (in 2D).

**Reminder:** Hierarchical clustering does not require you to set the number of clusters beforehand, the dendrogram helps you decide based on data structure.

In [ ]:
# Analyze dendrograms and choose optimal number of clusters
# Looking for the largest vertical distance without a horizontal line crossing it

# For each linkage method, extract clusters and visualize
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# COMPLETE LINKAGE ANALYSIS
print("COMPLETE LINKAGE ANALYSIS:")
print("Looking at dendrogram, major jumps occur around:")
print("- Distance ~12-13: Last large jump, suggesting 2-3 main clusters")
print("- Distance ~8-9: Another significant jump")
print("\nChosen: 2 clusters (balance between granularity and interpretability)")

n_clusters_complete = 2
clusters_complete = fcluster(Z_complete, n_clusters_complete, criterion='maxclust')

# Plot complete linkage dendrogram with threshold
ax = axes[0, 0]
dendrogram(Z_complete, ax=ax, leaf_rotation=90, leaf_font_size=8)
ax.axhline(y=12, c='red', linestyle='--', linewidth=2, label=f'Cut for {n_clusters_complete} clusters')
ax.set_title(f'Complete Linkage (n_clusters={n_clusters_complete})', fontsize=12, fontweight='bold')
ax.set_ylabel('Distance')
ax.legend()
ax.grid(True, alpha=0.3)

# Visualize clusters in 2D (using first 2 PCs for visualization)
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_sample_scaled)

ax = axes[0, 1]
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_complete, cmap='viridis', s=50, alpha=0.6)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
ax.set_title(f'Clusters - Complete Linkage (2D PCA)')
plt.colorbar(scatter, ax=ax)
ax.grid(True, alpha=0.3)

# AVERAGE LINKAGE ANALYSIS
print("\n\nAVERAGE LINKAGE ANALYSIS:")
print("Looking at dendrogram, major jumps occur around:")
print("- Distance ~13: Last large vertical jump")
print("- Suggests 2-3 main clusters")
print("\nChosen: 2 clusters")

n_clusters_average = 2
clusters_average = fcluster(Z_average, n_clusters_average, criterion='maxclust')

ax = axes[0, 2]
dendrogram(Z_average, ax=ax, leaf_rotation=90, leaf_font_size=8)
ax.axhline(y=13, c='red', linestyle='--', linewidth=2, label=f'Cut for {n_clusters_average} clusters')
ax.set_title(f'Average Linkage (n_clusters={n_clusters_average})', fontsize=12, fontweight='bold')
ax.set_ylabel('Distance')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_average, cmap='plasma', s=50, alpha=0.6)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
ax.set_title(f'Clusters - Average Linkage (2D PCA)')
plt.colorbar(scatter, ax=ax)
ax.grid(True, alpha=0.3)

# SINGLE LINKAGE ANALYSIS
print("\n\nSINGLE LINKAGE ANALYSIS:")
print("Looking at dendrogram, note the 'chaining' effect")
print("- Significant jump at distance ~14")
print("- Suggests 2-3 main clusters")
print("\nChosen: 2 clusters (despite chaining effect)")

n_clusters_single = 2
clusters_single = fcluster(Z_single, n_clusters_single, criterion='maxclust')

ax = axes[1, 1]
dendrogram(Z_single, ax=ax, leaf_rotation=90, leaf_font_size=8)
ax.axhline(y=14, c='red', linestyle='--', linewidth=2, label=f'Cut for {n_clusters_single} clusters')
ax.set_title(f'Single Linkage (n_clusters={n_clusters_single})', fontsize=12, fontweight='bold')
ax.set_ylabel('Distance')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1, 2]
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_single, cmap='coolwarm', s=50, alpha=0.6)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
ax.set_title(f'Clusters - Single Linkage (2D PCA)')
plt.colorbar(scatter, ax=ax)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("JUSTIFICATION FOR CHOSEN CLUSTER NUMBERS:")
print("="*60)
print("\nFor all three linkage methods, we chose 2 clusters because:")
print("1. The dendrograms show the most significant vertical distance jump")
print("2. Aligns with the binary nature of our fraud detection problem")
print("3. Provides interpretable clusters (normal vs anomalies)")
print("\nLinkage Comparison:")
print(f"- Complete Linkage: {len(np.unique(clusters_complete))} unique clusters created")
print(f"- Average Linkage: {len(np.unique(clusters_average))} unique clusters created")
print(f"- Single Linkage: {len(np.unique(clusters_single))} unique clusters created")
print("\nAverage linkage generally provides the best balance of")
print("compactness and separation for this dataset.")